email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from dataclasses import dataclass

#EmailContext is a simple dataclass that holds the user's email address and password. It defaults to "
@dataclass
class EmailContext:
    email_address: str = "johndoe@example.com"
    password: str = "pa$$word123!"

In [ ]:
from langchain.agents import AgentState
# AuthenticatedState is a simple dataclass that holds the user's authentication status. It defaults to False, indicating that the user is not authenticated.
class AuthenticatedState(AgentState):
    authenticated: bool

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# Tool call functions that simulate checking an inbox, sending an email, and authenticating a user. 
# These functions are decorated with the @tool decorator, which allows them to be used as tools in a LangChain agent. 
# The check_inbox function returns a sample email message, the send_email function simulates sending an email and returns a confirmation message, 
# and the authenticate function checks the provided email and password against the context and returns a Command indicating whether authentication was successful or not.
# Tool annotations provide metadata about the tool, such as its name, description, and input/output types.
@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """
# Send Email tool that simulates sending an email. It takes in the recipient's email address, subject, and body of the email as parameters and returns a confirmation message indicating that the email was sent.
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"
# Authenticate tool that checks the provided email and password against the context and returns a Command indicating whether authentication was successful or not.
@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

# callable that selects which tools are available based on the user's authentication status.
# Internal users have access to all tools, while external users only have access to the authenticate tool.
# The middleware modifies the request to include the appropriate tools before passing it to the handler for processing

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

# Middleware that generates a system prompt based on the user's authentication status.
authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Here’s what I found in your inbox:

From: Jane <jane@example.com>
Body: Hi Julie, I'm going to be in town next week and was wondering if we could grab a coffee? - best, Jane (jane@example.com)

Would you like me to reply? I can send one of these drafts, or tailor it to your style. I can also propose specific times or a meeting place.

Draft options (you can tell me which to send, or ask for changes):

Option 1 — warm and open-ended
Subject: Re: Coffee next week
Hi Jane,
That sounds perfect! I’d love to catch up. How about Tuesday at 10:00 AM or Wednesday afternoon? Let me know what works for you and where you’d like to meet.
Looking forward to it!
Julie

Option 2 — specific times
Subject: Re: Coffee next week
Hi Jane,
Great idea! I’m free Tuesday morning around 10:00 AM or Thursday afternoon around 4:00 PM. Do either of those times work? If not, feel free to suggest another time.
Best,
Julie

If you want, I can tailor the tone, add a location, or pick a single concrete time. Also tell 

In [9]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,
That sounds perfect! I’d love to catch up. How about Tuesday at 10:00 AM or Wednesday afternoon? Let me know what works for you and where you’d like to meet.
Looking forward to it!
Julie


In [11]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email sent to jane@example.com with subject Re: Coffee next week and body Hi Jane,
That sounds perfect! I’d love to catch up. How about Tuesday at 10:00 AM or Wednesday afternoon? Let me know what works for you and where you’d like to meet.
Looking forward to it!
Julie


In [12]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='7b28fd5c-8e89-44a2-98b3-71757d78ec2b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 222, 'prompt_tokens': 151, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EW32V1z25xcFfBu30NZdFX5Fw2JIT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a11235-f09f-7890-bfe8-254c5dc683f3-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_Ombk9c1VqG76BGqFX